# Đánh giá mô hình NLI: Base Model vs. Fine-tuned BamiBERT

- **Thư mục root**: `ML_final`
- **Dataset đánh giá**: `src/db/eval/vlsp_nli.parquet`
- **Mô hình gốc (Base)**: `Qualcomm-AI-Research/BamiBERT`
- **Mô hình Fine-tuned**: `src/be/NLIProvider/models/bamibert_vilegalnli`
- **Quy ước nhãn**: `choices = ['Có', 'Không']` -> `answer = 0` ('Có') tương ứng **Entailment (1)**, `answer = 1` ('Không') tương ứng **Non-entailment / Contradiction (0)**.

In [ ]:
import os
import sys
from pathlib import Path

# Đảm bảo ML_final nằm trong sys.path khi chạy notebook
CURRENT_DIR = Path.cwd()
for p in [CURRENT_DIR, *CURRENT_DIR.parents]:
    if p.name == "ML_final" or (p / "src" / "be").exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        os.chdir(str(p))
        break

print(f"📁 Thư mục làm việc hiện tại: {Path.cwd()}")

In [ ]:
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, classification_report

from src.be.config.config import config

# Thiết lập đường dẫn tương đối từ ML_final root
DATASET_PATH = Path("src/db/eval/vlsp_nli.parquet")
FINETUNED_MODEL_PATH = Path("src/be/NLIProvider/models/bamibert_vilegalnli")
BASE_MODEL_PATH = "Qualcomm-AI-Research/BamiBERT"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🎯 Thiết bị thực thi: {device}")
print(f"📊 Dataset: {DATASET_PATH}")
print(f"🤖 Fine-tuned Model: {FINETUNED_MODEL_PATH}")

## 1. Nạp và xử lý tập dữ liệu kiểm thử

In [ ]:
# Đọc dataset từ src/db/eval/vlsp_nli.parquet
df = pd.read_parquet(DATASET_PATH)

# Ánh xạ nhãn: answer 0 ('Có') -> 1 (Entailment), answer 1 ('Không') -> 0 (Non-entailment)
if "choices" in df.columns:
    df["nli_label"] = df["answer"].apply(lambda x: 1 if x == 0 else 0)
else:
    df["nli_label"] = df["answer"]

label_col = "nli_label"

print(f"Tổng số mẫu kiểm thử: {len(df)}")
print("\nPhân phối nhãn (1: Entailment, 0: Non-entailment):")
print(df[label_col].value_counts())
print("\nMẫu dữ liệu đầu tiên:")
df[["specific_question", "legal_document", "answer", label_col]].head()

## 2. Hàm suy luận và đánh giá theo Batch

In [ ]:
def evaluate_model(model_name_or_path, df, batch_size=16, max_length=2048):
    print(f"\n>>> Đang nạp model và tokenizer từ: {model_name_or_path}")
    tokenizer = AutoTokenizer.from_pretrained(str(model_name_or_path))
    model = AutoModelForSequenceClassification.from_pretrained(
        str(model_name_or_path),
        num_labels=2,
        ignore_mismatched_sizes=True
    ).to(device)
    model.eval()
    
    predictions = []
    
    for i in tqdm(range(0, len(df), batch_size), desc=f"Đang đánh giá {Path(str(model_name_or_path)).name}"):
        batch = df.iloc[i : i + batch_size]
        premises = batch["specific_question"].tolist()
        hypotheses = batch["legal_document"].tolist()
        
        inputs = tokenizer(
            premises,
            hypotheses,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            
        preds = torch.argmax(logits, dim=-1).cpu().numpy()
        predictions.extend(preds)
        
    return predictions

## 3. Thực thi đánh giá Base Model vs Fine-tuned Model

In [ ]:
print("=== 1. ĐÁNH GIÁ BASE MODEL (BamiBERT) ===")
base_preds = evaluate_model(BASE_MODEL_PATH, df)

print("\n=== 2. ĐÁNH GIÁ FINE-TUNED MODEL (BamiBERT-ViLegalNLI) ===")
finetuned_preds = evaluate_model(FINETUNED_MODEL_PATH, df)

## 4. Tổng hợp và so sánh kết quả Metrics

In [ ]:
def calculate_metrics(y_true, y_pred):
    return {
        "Accuracy": round(accuracy_score(y_true, y_pred), 4),
        "Precision": round(precision_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
        "Recall": round(recall_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
        "F1-Score": round(f1_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
        "Macro F1": round(f1_score(y_true, y_pred, average="macro", zero_division=0), 4)
    }

y_true = df[label_col].tolist()

base_metrics = calculate_metrics(y_true, base_preds)
finetuned_metrics = calculate_metrics(y_true, finetuned_preds)

comparison_df = pd.DataFrame([
    {"Model": "Base Model (BamiBERT)", **base_metrics},
    {"Model": "Fine-tuned Model (BamiBERT-ViLegalNLI)", **finetuned_metrics}
])

print("\n" + "=" * 65)
print("                    BẢNG TỔNG HỢP SO SÁNH")
print("=" * 65)
print(comparison_df.to_string(index=False))

print("\n" + "-" * 65)
print("Chi tiết Classification Report: Base Model")
print("-" * 65)
print(classification_report(y_true, base_preds, target_names=["Non-entailment (0)", "Entailment (1)"], zero_division=0))

print("-" * 65)
print("Chi tiết Classification Report: Fine-tuned Model")
print("-" * 65)
print(classification_report(y_true, finetuned_preds, target_names=["Non-entailment (0)", "Entailment (1)"], zero_division=0))